In [125]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv
/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv
/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv


In [126]:
train = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv')
test = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv')

print(train.shape)
train.head()

(6818, 13)


,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,HEALTH AND HYGIENE,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85
3,row_00003,PRD-UN5Z3J,NaN,Regular,0.0449,soft drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36


In [127]:
print(train.columns.tolist())

['id', 'product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'total_sales']


In [128]:
print(train.isnull().sum())
print(train.dtypes)



id                        0
product_code              0
product_weight_kg      1225
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size             1919
store_location_tier       0
store_format              0
total_sales               0
dtype: int64
id                      object
product_code            object
product_weight_kg      float64
fat_content             object
shelf_visibility       float64
product_category        object
product_price          float64
store_code              object
store_age_years          int64
store_size              object
store_location_tier     object
store_format            object
total_sales            float64
dtype: object


In [129]:
print(train['fat_content'].unique())
print(train['product_category'].unique())
print(train['store_size'].unique())
print(train['store_location_tier'].unique())
print(train['store_format'].unique())

['Low Fat' 'Regular']
['Frozen Foods' 'HEALTH AND HYGIENE' 'Canned' 'soft drinks' 'meat'
 'Snack Foods' 'baking goods' 'household' 'Household' 'Dairy' 'dairy'
 'CANNED' 'SNACK FOODS' 'HOUSEHOLD' 'Soft Drinks' 'Fruits and Vegetables'
 'Breakfast' 'Breads' 'Meat' 'BAKING GOODS' 'canned' 'DAIRY' 'breakfast'
 'Baking Goods' 'Health and Hygiene' 'MEAT' 'Seafood'
 'fruits and vegetables' 'frozen foods' 'snack foods' 'FROZEN FOODS'
 'Others' 'FRUITS AND VEGETABLES' 'health and hygiene' 'OTHERS' 'BREADS'
 'SEAFOOD' 'SOFT DRINKS' 'Hard Drinks' 'breads' 'starchy foods'
 'hard drinks' 'HARD DRINKS' 'others' 'Starchy Foods' 'seafood'
 'BREAKFAST' 'STARCHY FOODS']
['Large' 'Small' 'Medium' nan]
['Tier_3' 'Tier_1' 'Tier_2']
['Standard Supermarket' 'Flagship Hypermarket' 'Superstore' 'Corner Shop']


In [130]:
train['product_category'] = train['product_category'].str.lower().str.strip()
test['product_category'] = test['product_category'].str.lower().str.strip()

print(train['product_category'].nunique())
print(train['product_category'].unique())

16
['frozen foods' 'health and hygiene' 'canned' 'soft drinks' 'meat'
 'snack foods' 'baking goods' 'household' 'dairy' 'fruits and vegetables'
 'breakfast' 'breads' 'seafood' 'others' 'hard drinks' 'starchy foods']


In [131]:
train['product_weight_kg']= train['product_weight_kg'].fillna(train['product_weight_kg'].median())
test['product_weight_kg'] = test['product_weight_kg'].fillna(train['product_weight_kg'].median())

train['store_size'] = train['store_size'].fillna('Missing')
test['store_size'] = test['store_size'].fillna('Missing')

print(train.isnull().sum())

id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
total_sales            0
dtype: int64


In [132]:
print(train['total_sales'].describe())

count     6818.000000
mean      2174.756574
std       1697.894956
min         32.700000
25%        834.792500
50%       1790.890000
75%       3092.587500
max      12996.820000
Name: total_sales, dtype: float64


In [133]:
print(train[['product_weight_kg','shelf_visibility','product_price','store_age_years','total_sales']].corr()['total_sales'])

product_weight_kg    0.011367
shelf_visibility    -0.118124
product_price        0.569833
store_age_years      0.039443
total_sales          1.000000
Name: total_sales, dtype: float64


In [134]:
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error 
import numpy as np

train['is_train'] = 1
test['is_train'] = 0
test['total_sales'] = 0

combined = pd.concat([train,test], axis=0)

categorical_cols = ['fat_content','product_category','store_size','store_location_tier','store_format']
combined_encoded = pd.get_dummies(combined,columns = categorical_cols)

train_encoded = combined_encoded[combined_encoded['is_train']== 1].drop(['is_train'], axis=1)
test_encoded = combined_encoded[combined_encoded['is_train'] == 0].drop(['is_train','total_sales'], axis=1)

print(train_encoded.shape)
print(test_encoded.shape)



(6818, 37)
(1705, 36)


In [135]:
X = train_encoded.drop(['id','product_code','store_code','total_sales'], axis = 1)
y = train_encoded['total_sales']

X_test_final = test_encoded.drop(['id','product_code','store_code'], axis=1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)

model = LinearRegression()
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
print("Validation RMSE:" ,rmse)

Validation RMSE: 1122.556271087588


In [136]:
print((train['total_sales'].mean()))

2174.7565737753007


In [137]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(y_val,rf_val_preds))
print("Random Forest Validation RMSE:",rf_rmse)

Random Forest Validation RMSE: 1115.5282212284328


In [138]:
importances = pd.Series(rf_model.feature_importances_,index=X_train.columns)
print(importances.sort_values(ascending=False).head(15))

product_price                             0.435979
store_format_Corner Shop                  0.191608
shelf_visibility                          0.094097
product_weight_kg                         0.074088
store_age_years                           0.045329
store_format_Flagship Hypermarket         0.043322
product_category_fruits and vegetables    0.008869
product_category_snack foods              0.007640
product_category_household                0.007017
product_category_dairy                    0.006312
product_category_frozen foods             0.006151
product_category_soft drinks              0.005676
store_size_Small                          0.005605
fat_content_Low Fat                       0.005575
fat_content_Regular                       0.005370
dtype: float64


In [139]:
!pip install xgboost -q
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
xgb_model.fit(X_train,y_train)

xgb_val_preds = xgb_model.predict(X_val)
xgb_rmse = np.sqrt(mean_squared_error(y_val,xgb_val_preds))

print("XGBoost Validation RMSE:", xgb_rmse)

XGBoost Validation RMSE: 1114.3664330344254


In [140]:
X_train['price_x_cornershop'] = X_train['product_price'] * X_train['store_format_Corner Shop']
X_val['price_x_cornershop'] = X_val['product_price'] * X_val['store_format_Corner Shop']
X_test_final['price_x_cornershop'] = X_test_final['product_price'] * X_test_final['store_format_Corner Shop']

xgb_model2 = XGBRegressor(n_estimators = 300, learning_rate = 0.05, max_depth = 5, random_state =42)
xgb_model2.fit(X_train,y_train)
preds2 =xgb_model2.predict(X_val)
rmse2 = np.sqrt(mean_squared_error(y_val, preds2))

print("XGBoost + interaction RMSE:" ,rmse2)

XGBoost + interaction RMSE: 1109.4558862651286


In [141]:
import numpy as np
print(train['total_sales'].skew())

1.1539947519138116


In [142]:
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

xgb_model3 = XGBRegressor(n_estimators = 300, learning_rate = 0.05, max_depth = 5, random_state = 42)
xgb_model3.fit(X_train, y_train_log)

preds3_log = xgb_model3.predict(X_val)
preds3 = np.expm1(preds3_log)

rmse3 = np.sqrt(mean_squared_error(y_val, preds3))
print("XGBoost + log-target RMSE", rmse3)

XGBoost + log-target RMSE 1142.2898179569352


In [143]:
sample_sub = pd.read_csv ('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv')

In [144]:
test_preds = xgb_model2.predict(X_test_final)

submission = pd.DataFrame({
    'id': test["id"],
    'total_sales': test_preds
})

submission.to_csv('submission.csv', index = False)
print(submission.head())

          id  total_sales
0  row_00009  2728.875244
1  row_00015  5280.588379
2  row_00019  3399.066406
3  row_00020  2988.871582
4  row_00023  2422.150879


In [145]:
sample_sub = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv')
print(sample_sub.columns.tolist())
print(submission.columns.tolist())

['id', 'total_sales']
['id', 'total_sales']


In [146]:
rf_test_preds = rf_model.predict(X_test_final.drop('price_x_cornershop',axis=1))
xgb_test_preds = xgb_model2.predict(X_test_final)

blended_preds = (rf_test_preds + xgb_test_preds)/2

submission2 = pd.DataFrame({'id': test['id'],'total_sales': blended_preds})
submission2.to_csv('submission2.csv', index= False)
print(submission2.head())

          id  total_sales
0  row_00009  2823.857597
1  row_00015  5286.591589
2  row_00019  3565.488378
3  row_00020  2972.401716
4  row_00023  2654.778739


In [147]:
store_avg = train.groupby('store_code')['total_sales'].mean().rename('store_avg_sales')
category_avg = train.groupby('product_category')['total_sales'].mean().rename('category_avg_sales')

train_full = train.loc[X_train.index].copy()
val_full = train.loc[X_val.index].copy()

X_train['store_avg_sales'] = train_full['store_code'].map(store_avg)
X_val['store_avg_sales'] = val_full['store_code'].map(store_avg)
X_test_final['store_avg_sales'] = test['store_code'].map(store_avg)

X_train['category_avg_sales'] = train_full['product_category'].map(category_avg)
X_val['category_avg_sales'] = val_full['product_category'].map(category_avg)
X_test_final['category_avg_sales'] = test['product_category'].map(category_avg)

xgb_model4 = XGBRegressor(n_estimators = 300, learning_rate=0.05, max_depth=5, random_state=42)
xgb_model4.fit(X_train,y_train)

preds4 = xgb_model4.predict(X_val)
rmse4 = np.sqrt(mean_squared_error(y_val,preds4))
print("XGBoost + group averages RMSE:", rmse4)

XGBoost + group averages RMSE: 1119.839022531088


In [148]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [200,300,400,500],
    'max_depth':[3,4,5,6,7],
    'learning_rate':[0.01,0.03,0.05,0.07,0.1],
    'subsample': [0.7,0.8,0.9,1.0],
    'colsample_bytree':[0.7,0.8,0.9,1.0]
}

xgb_base = XGBRegressor(random_state=42)

random_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV RMSE:", -random_search.best_score_)

Best params: {'subsample': 0.9, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best CV RMSE: 1087.5251999040286


In [149]:
final_model = XGBRegressor(
    n_estimators = 500,
    max_depth =4,
    learning_rate = 0.01,
    subsample = 0.9,
    colsample_bytree = 0.7,
    random_state = 42
)
final_model.fit(X_train, y_train)

final_val_preds = final_model.predict(X_val)
final_rmse = np.sqrt(mean_squared_error(y_val, final_val_preds))
print("Final tuned model RMSE:", final_rmse)

Final tuned model RMSE: 1075.8687052101106


In [150]:
final_test_preds = final_model.predict(X_test_final)

submission3 = pd.DataFrame({'id': test['id'], 'total_sales': final_test_preds})
submission3.to_csv('submission3.csv', index=False)
print(submission3.head())

          id  total_sales
0  row_00009  2729.195801
1  row_00015  4309.998535
2  row_00019  3297.598389
3  row_00020  2922.182129
4  row_00023  2460.098145


In [151]:
!pip install lightgbm -`
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(n_estimators=500, max_depth=4,learning_rate = 0.01, subsample =0.9, colsample_bytree= 0.7,random_state =42)
lgbm_model.fit(X_train, y_train)

lgbm_val_preds = lgbm_model.predict(X_val)
lgbm_rmse = np.sqrt(mean_squared_error(y_val, lgbm_val_preds))
print("LightGBM RMSE:", lgbm_rmse)

/bin/bash: -c: line 1: unexpected EOF while looking for matching ``'
/bin/bash: -c: line 2: syntax error: unexpected end of file
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1091
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 36
[LightGBM] [Info] Start training from score 2163.170056
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

In [152]:
category_price_avg = train.groupby('product_category')['product_price'].mean().rename('category_price_avg')

X_train['price_vs_category_avg'] = train_full['product_price']/ train_full['product_category'].map(category_price_avg)
X_val['price_vs_category_avg'] = val_full['product_price']/val_full['product_category'].map(category_price_avg)
X_test_final['price_vs_category_avg'] = test['product_price']/ test['product_category'].map(category_price_avg)

final_model2 = XGBRegressor(n_estimators=500, max_depth=4, learning_rate= 0.01, subsample=0.9, colsample_bytree=0.7, random_state=42)
final_model2.fit(X_train, y_train)

preds5 = final_model2.predict(X_val)
rmse5 =  np.sqrt(mean_squared_error(y_val, preds5))
print("XGBoost + price-vs-category RMSE:", rmse5)

XGBoost + price-vs-category RMSE: 1074.5956902065147


In [153]:
final_test_preds2 = final_model2.predict(X_test_final)

submission4 = pd.DataFrame({'id': test['id'], 'total_sales': final_test_preds2})
submission4.to_csv('submission4.csv', index=False)
print(submission4.head())

          id  total_sales
0  row_00009  2708.777100
1  row_00015  4113.436035
2  row_00019  3250.462158
3  row_00020  2983.236328
4  row_00023  2475.125244
